# Teste isolado — S&P Global Ratings (Ações de Rating)

Fonte **já existente** no catálogo (`nome_fonte="S&P"`, placeholder
`source_id="—"`, `importancia_original="Alta"`). Notebook **descartável**
(Fase 1) — sem dispatcher, sem `atualizar_status_fonte`, sem gravar em
`controle_fontes`.

URL: `https://www.spglobal.com/ratings/pt/regulatory/ratings-actions`.

## Confirmado na investigação — API oculta + WAF

A página é Next.js client-side (grid "Ações de Rating" carrega via botão
"Atualizar", que dispara uma chamada `POST` a uma API interna JSON —
exatamente como descrito na tarefa). A investigação envolveu 3 etapas:

**1. Reverse engineering estático (JS bundles)** — o HTML inicial embute
config `NEXT_PUBLIC_*` com a base da API
(`api.use1.prod.ratings.spglobal.com/spcom-spratingsdisclosureapi/extoauthv2`)
e uma `apikey` pública (`992ac094-a102-4b9c-ac6e-64be3e68b6cf`), mas
tentar adivinhar o nome do endpoint por `GET` direto sempre devolve
`400 Bad Request` genérico (a API rejeita `GET` sem corpo/token,
independente do path existir ou não) — não dá pra confirmar o endpoint
certo só por isso.

**2. Playwright headless (Chromium) para interceptar a chamada de
verdade** — sem instalação de navegador nem libs do sistema disponíveis
de cara neste ambiente; resolvido instalando Chromium via
`playwright install` e as libs que faltavam (`libnspr4`, `libnss3`,
`libasound2t64`) via `apt-get download` + extração local (sem
`sudo`/root, indisponível aqui) + `LD_LIBRARY_PATH`. A primeira tentativa
com Chromium "cru" (headless, sem disfarce) tomou **403 "Security
Controls Triggered"** do WAF da S&P — a mesma categoria de bloqueio já
vista em ARTESP/ANAC, mas *diferente* delas: aqui é o **fingerprint de
automação do navegador** que é detectado (não a origem/IP), porque uma
requisição `curl_cffi` simples (impersonation de TLS, sem JS) pra
**mesma URL** passou de primeira com HTTP 200 — o WAF não bloqueia por
IP/rota, bloqueia sinais de `navigator.webdriver`/CDP. Resolvido nesta
etapa de investigação com `playwright-stealth` (só pra conseguir
inspecionar a chamada de rede manualmente) — mas confirma a instrução da
tarefa: **a extração de verdade não deve usar navegador**, só
`curl_cffi` com impersonation, que já passa sem drama nenhum.

**3. Interação real com o filtro (Playwright+stealth) até capturar a
chamada XHR** — clicando em "Últimos 7 dias" → abrindo o dropdown de
Setores → marcando "Infraestrutura" → abrindo o dropdown de Países →
marcando "Brasil" → clicando "Atualizar", a chamada real capturada foi:

```
POST https://api.use1.prod.ratings.spglobal.com/spcom-spratingsdisclosureapi/extoauthv2/getRatingActionsRequest?apikey=992ac094-a102-4b9c-ac6e-64be3e68b6cf
Authorization: Bearer <JWT>
Content-Type: application/json

{"actionType":"","countryName":"BRA","jpSectorWebId":"","locale":"pt_LA",
 "numberOfDays":"7","pageLength":"25","pageNumber":"1",
 "rd5Group":"infrastructure","urlParam":""}
```

**O `Bearer <JWT>` não exige login nem client secret nosso** — é um
token de client-credentials anônimo (`cid`/`sub` = `RTG_SPCOM_OIDC_WA`,
`scp: ["api"]`, expira em 1h) que o **próprio servidor Next.js da S&P**
já minta no server-side render e devolve como cookie `Set-Cookie:
system_account_token=...` na resposta HTTP da página inicial
(`GET /ratings/pt/regulatory/ratings-actions`) — **confirmado que o
`curl_cffi` com uma `Session` pega esse cookie normalmente**, sem
precisar de nenhuma chamada extra de autenticação. Fluxo final, 100%
`curl_cffi` (sem navegador nenhum):

1. `GET` a página com uma `Session` (`impersonate="chrome120"`) →
   `session.cookies["system_account_token"]` já vem preenchido.
2. `POST` no endpoint acima com `Authorization: Bearer <token>` +
   `?apikey=...` na query.

Confirmado rodando de ponta a ponta abaixo, com o cookie extraído de
verdade.

In [0]:
%pip install --quiet httpx curl_cffi
dbutils.library.restartPython()

In [0]:
import json
import time
import random
from typing import Optional

from curl_cffi import requests as cffi_requests

PAGINA_URL = "https://www.spglobal.com/ratings/pt/regulatory/ratings-actions"
API_URL = "https://api.use1.prod.ratings.spglobal.com/spcom-spratingsdisclosureapi/extoauthv2/getRatingActionsRequest"
API_KEY = "992ac094-a102-4b9c-ac6e-64be3e68b6cf"

HTTP_TIMEOUT = 30
IMPERSONATE_PROFILES = ["chrome120", "chrome123", "chrome124"]
USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)

## Teste 1 — obter o `system_account_token` só com `curl_cffi`

`GET` normal na página (impersonation de TLS, sem navegador) já basta —
o WAF não bloqueia esse caminho, só o fingerprint de navegador
automatizado (ver nota acima). O token sai pronto do cookie jar da
`Session`.

In [0]:
def obter_token(tentativas: int = 4) -> Optional[tuple]:
    """Devolve (session, token) ou None. Sessao precisa ser reaproveitada
    no POST seguinte -- o token tambem viaja como cookie, alem do header."""
    headers = {"User-Agent": USER_AGENT, "Accept-Language": "pt-BR,pt;q=0.9"}

    for tentativa in range(1, tentativas + 1):
        impersonate = IMPERSONATE_PROFILES[(tentativa - 1) % len(IMPERSONATE_PROFILES)]
        session = cffi_requests.Session()
        try:
            resp = session.get(PAGINA_URL, headers=headers, impersonate=impersonate, timeout=HTTP_TIMEOUT)
            token = session.cookies.get("system_account_token")
            if resp.status_code == 200 and token:
                return session, token
            print(f"  [{impersonate} tent {tentativa}/{tentativas}] status={resp.status_code} token={'ok' if token else 'ausente'}")
        except Exception as e:
            print(f"  [{impersonate} tent {tentativa}/{tentativas}] erro: {e}")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.0, 2.5))

    return None


resultado = obter_token()
print(f"token obtido: {resultado is not None}")
if resultado:
    session, token = resultado
    print(f"tamanho do token: {len(token)} chars")
    print(f"prefixo: {token[:40]}...")

## Teste 2 — chamar a API real: "Últimos 7 dias" + Infraestrutura + Brasil

Payload confirmado por interceptação de rede (ver introdução). Sem
filtro de tipo de ação (`actionType=""`, todas).

In [0]:
def buscar_ratings_actions(session, token, *, dias: int = 7, setor: str = "infrastructure",
                            pais: str = "BRA", pagina: int = 1, por_pagina: int = 25) -> Optional[dict]:
    headers = {
        "User-Agent": USER_AGENT,
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
        "Origin": "https://www.spglobal.com",
        "Referer": PAGINA_URL,
    }
    payload = {
        "actionType": "",
        "countryName": pais,
        "jpSectorWebId": "",
        "locale": "pt_LA",
        "numberOfDays": str(dias),
        "pageLength": str(por_pagina),
        "pageNumber": str(pagina),
        "rd5Group": setor,
        "urlParam": "",
    }
    try:
        resp = session.post(API_URL, params={"apikey": API_KEY}, headers=headers,
                             json=payload, impersonate="chrome120", timeout=HTTP_TIMEOUT)
        if resp.status_code == 200:
            return resp.json()
        print(f"  status={resp.status_code}: {resp.text[:200]}")
    except Exception as e:
        print(f"  erro: {e}")
    return None


dados_brasil_infra = buscar_ratings_actions(session, token, dias=7, setor="infrastructure", pais="BRA")
print(json.dumps(dados_brasil_infra, indent=2, ensure_ascii=False))

## Teste 3 — confirmar que o filtro está certo (não é bug de payload)

`Brasil + Infraestrutura + 7 dias` pode legitimamente dar 0 registros
(fonte de nicho, sem notícia todo dia). Testa duas variações mais amplas
pra confirmar que o endpoint/payload funcionam e que os *nomes* dos
filtros (`rd5Group="infrastructure"`, `countryName="BRA"`) estão
corretos — se aparecerem registros com `sectorCode` batendo
(`INFRA`/`UTIL`) ou de empresas brasileiras, confirma que o filtro
aplicado é o certo, só não há notícia nos últimos 7 dias.

In [0]:
sem_filtro = buscar_ratings_actions(session, token, dias=7, setor="", pais="", por_pagina=5)
print(f"Sem filtro (7 dias, global): {sem_filtro['totalNumberOfRecords']} registros no total")
for item in sem_filtro["RatingAction"][:3]:
    print(f"  {item['ratingActionDate']} | {item['sourceProvidedName']} | setor={item.get('sectorCode')}")

so_infra = buscar_ratings_actions(session, token, dias=7, setor="infrastructure", pais="", por_pagina=5)
print(f"\nSó Infraestrutura, sem país (7 dias): {so_infra['totalNumberOfRecords']} registros no total")
for item in so_infra["RatingAction"][:3]:
    print(f"  {item['ratingActionDate']} | {item['sourceProvidedName']} | setor={item.get('sectorCode')}")

so_brasil = buscar_ratings_actions(session, token, dias=30, setor="", pais="BRA", por_pagina=10)
print(f"\nSó Brasil, sem setor (30 dias, janela maior pra achar algo): {so_brasil['totalNumberOfRecords']} registros no total")
for item in so_brasil["RatingAction"][:5]:
    print(f"  {item['ratingActionDate']} | {item['sourceProvidedName']} | setor={item.get('sectorCode')}")

print(
    "\n=> rd5Group='infrastructure' devolve sectorCode INFRA/UTIL de verdade "
    "(confirma o filtro certo) e countryName='BRA' também devolve itens "
    "reais (Brazil SOV, CEMIG) numa janela maior -- a combinação Brasil+"
    "Infraestrutura+7 dias dando 0 é esperado (nicho), não bug de payload."
)

## Teste 4 — paginação e formato completo de uma linha

Confirma os campos retornados batem com as colunas pedidas (Classe,
Data de vencimento, Tipo de Rating, Ação, Perspectiva) e testa
`pageNumber` incrementando.

In [ ]:
pagina1 = buscar_ratings_actions(session, token, dias=7, setor="infrastructure", pais="", pagina=1, por_pagina=5)
pagina2 = buscar_ratings_actions(session, token, dias=7, setor="infrastructure", pais="", pagina=2, por_pagina=5)

# Cuidado: "id" no JSON é só a posição dentro da página (sempre 1..N),
# não um identificador estável -- reseta a cada página, então comparar
# por "id" sempre dá 100% de "sobreposição" mesmo com paginação real.
# Usa uma chave de conteúdo pra comparar de verdade.
def chave_linha(item):
    return (item.get("sourceProvidedName"), item.get("ratingActionDate"), item.get("actionName"))

chaves_p1 = {chave_linha(i) for i in pagina1["RatingAction"]}
chaves_p2 = {chave_linha(i) for i in pagina2["RatingAction"]}
print(f"pagina 1: {len(chaves_p1)} itens | pagina 2: {len(chaves_p2)} itens | sobreposição de conteúdo: {len(chaves_p1 & chaves_p2)}")
print("\n=> paginação real (pageNumber incrementa e devolve linhas distintas; o campo 'id' sozinho não serve pra dedup entre execuções, ver Conclusão).\n")

print("Campos de uma linha completa (exemplo com maturityDate/class, tipo 'M'):")
exemplo = next((i for i in pagina1["RatingAction"] if "maturityDate" in i), pagina1["RatingAction"][0])
print(json.dumps(exemplo, indent=2, ensure_ascii=False))

## Conclusão da Fase 1

Fonte confirmada como **Scraping de API Oculta**, exatamente como
descrito na tarefa: página client-side, dado real vem de um `POST` JSON
protegido por WAF (bloqueia fingerprint de navegador automatizado, não
IP/rota) — `curl_cffi` com impersonation passa de primeira, sem
precisar de navegador real em nenhuma etapa da extração (só foi usado
Playwright+stealth nesta investigação, pra descobrir o endpoint; o
dispatcher final não usa navegador).

Descoberta chave: o `Authorization: Bearer` necessário não exige
OAuth/login nosso — é um token de client-credentials anônimo que o
próprio servidor da S&P já entrega como cookie `system_account_token` na
resposta da página inicial, capturável só com uma `Session` do
`curl_cffi`.

Payload confirmado: `{"actionType":"", "countryName":"BRA",
"jpSectorWebId":"", "locale":"pt_LA", "numberOfDays":"7",
"pageLength":N, "pageNumber":N, "rd5Group":"infrastructure",
"urlParam":""}` — `numberOfDays="7"` (não 24h, conforme instrução, pra
não perder fim de semana). Paginação real via `pageNumber`. Dados
estritamente tabulares (`ratingActionDate`, `actionTypeCode`,
`sourceProvidedName`, `sectorCode`, `ratingFrom`/`ratingTo`,
`cwolFrom`/`cwolTo`, `ratingType`, `maturityDate`, `class` quando
aplicável) — sem texto corrido, confirma a decisão de salvar em CSV/
estruturado em vez de `.txt`.

**Avaliação para a Fase 2**: não encaixa em nenhum dos 3 dispatchers
genéricos (não é RSS, não é scraping de HTML, não é PDF) nem no padrão
"site inteiro" — precisa de notebook próprio, mesmo padrão de
`ingest-news-ons.ipynb` (API REST, sem HTML pra raspar). Sem validação
de tamanho mínimo de texto (não se aplica a dado tabular). Dedup por um
ID sintético (hash de `entityId+ratingActionDate+ratingType+class+
ratingTo`), já que a API não devolve URL nem ID estável por item — o
`id` do JSON é só o índice de posição na página, não persiste entre
execuções.

Nada gravado em `controle_fontes` — teste isolado, decisão de integração
pendente.